# Single-cell RNA-seq: PCA and k-means clustering

*bioinformatics · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/bioinformatics/single_cell_pca)

A synthetic single-cell experiment: cells of a few types, each switching on its own marker genes, with negative-binomial counts and cells sequenced to different depths. We run the standard pipeline as tensor operations: normalise to $\log(1 + 10^4 K_{cg}/n_c)$, keep highly variable genes, PCA by SVD, then k-means with several restarts in parallel, and score the clusters against the true types with the adjusted Rand index.

**Vector form**

$$X_{cg} = \log\!\Big(1 + 10^4\,\frac{K_{cg}}{n_c}\Big),\qquad \mathbf Z = \mathbf X - \bar{\mathbf X} = \mathbf U\,\boldsymbol\Sigma\,\mathbf V^{\mathsf T},\qquad \min_{\{\boldsymbol\mu_k\}} \sum_c \min_k \lVert \mathbf y_c - \boldsymbol\mu_k\rVert^2$$

**Index (tensor) form**

$$n_c = K_{cg}\,\mathbb 1_g \qquad y_{cp} = Z_{cg}\,V_{gp} \qquad d_{rck} = y_{cp}y_{cp} - 2\,y_{cp}\,\mu_{rkp} + \mu_{rkp}\mu_{rkp} \qquad \mu_{rkp} = \frac{A_{rck}\,y_{cp}}{A_{rck}\,\mathbb 1_c}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\mu_{kg}$ | `type_profiles(G, K, M, fold_change, seed)` | (K, G) | mean expression of gene $g$ in type $k$: a log-normal baseline, times the fold change for the $M$ marker genes of $k$ |
| $K_{cg}$ | `poisson(Gamma(1/φ, mean s_c μ_{k(c)g}))` | (C, G) | negative-binomial counts (variance $m + \varphi m^2$); $s_c$ is a log-normal sequencing depth |
| $n_c$ | `einsum('cg->c', counts)` | (C,) | library size: total counts of cell $c$, $g$ summed |
| $X_{cg}$ | `log1p(1e4 * counts / n_c[:, None])` | (C, G) | depth-normalised (counts per $10^4$, like CPM) and log-transformed |
| $z_g$ | `hvg_scores(scaled)` | (G,) | dispersion $\mathrm{Var}/\mathrm{Mean}$, standardised within bins of similar mean (bin sums by one-hot einsum) |
| $U\Sigma V^{\mathsf T}$ | `torch.linalg.svd(Z, full_matrices=False)` | (C, H), (H,), (H, H) | PCA of the centred matrix on the $H$ highly variable genes; torch.pca_lowrank gives the same top components |
| $y_{cp}$ | `einsum('cg,gp->cp', Z, V)` | (C, P) | PC scores, $g$ summed; $\Sigma_p^2/\sum_q\Sigma_q^2$ is the explained-variance ratio |
| $d_{rck}$ | `sq_distances(y, mu)` | (R, C, K) | squared distance of cell $c$ to centre $k$ of restart $r$; $p$ summed, all $R$ restarts at once |
| $A_{rck}$ | `one_hot(d.argmin(-1), K)` | (R, C, K) | assignment step: each cell joins its nearest centre |
| $\mu_{rkp}$ | `einsum('rck,cp->rkp', A, y) / einsum('rck->rk', A)` | (R, K, P) | update step: each centre moves to the mean of its cells, $c$ summed |
| $\mathrm{ARI}$ | `adjusted_rand_index(pred, truth)` | () | pair-counting agreement from the contingency table $n_{ij}$ (one-hot einsum); 1 = perfect, about 0 = random |

Each cell yields a few thousand mRNA counts spread over thousands of genes, so most entries are zero and the raw counts mostly reflect how deeply a cell was sequenced. Dividing by the library size $n_c$ removes the depth; the log tames the huge range of expression levels. Most genes carry only noise, so we keep the genes whose variance exceeds what genes of similar mean show: the highly variable genes, enriched in the planted markers. PCA then compresses them: with $K$ types the signal lives in about $K-1$ directions, visible as the first $K-1$ bars of the scree plot standing above a flat noise floor. k-means alternates two steps, assign each cell to its nearest centre and move each centre to the mean of its cells; neither step can increase the inertia $\sum_c \lVert y_c - \mu_{k(c)}\rVert^2$, so it converges, but only to a local minimum, which is why several random starts are run side by side (grey curves) and the best is kept. Because we know the true types, the adjusted Rand index scores the result honestly. The optional t-SNE map is a small force layout trained with autograd (Student-t similarities, early exaggeration); distances between clusters in such maps mean little. Real pipelines (Scanpy, Seurat) add quality control, batch correction and graph-based clustering (Leiden) on a nearest-neighbour graph, and real cell types are rarely this clean. Try a smaller fold change, fewer markers or a wrong $k$.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/bioinformatics/single_cell_pca/main.py
"""Main loop: synthetic counts → normalise → highly variable genes → PCA → k-means (→ t-SNE) → ARI → plot."""

import time
from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import adjusted_rand_index, hvg_scores, normalise, pca

SEED = 0
N_INIT = 8  # k-means restarts, run side by side


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    fast = fast_mode()
    t0 = time.perf_counter()
    n_cells = 300 if fast else params["n_cells"]

    # --- a synthetic experiment: K cell types, each with its own marker genes ---
    mu, markers = tensors.type_profiles(
        params["n_genes"], params["n_types"], params["n_markers"], params["fold_change"], SEED, dev
    )  # (K, G), (K, M)
    counts, labels, _ = tensors.sample_counts(n_cells, mu, params["dispersion"], params["depth_spread"], SEED + 1)
    lib = counts.sum(1)
    zeros = (counts == 0).float().mean().item()
    print(f"{n_cells} cells × {params['n_genes']} genes, {params['n_types']} cell types; library sizes "
          f"{lib.min():.0f} … {lib.max():.0f} (median {lib.median():.0f}); {zeros:.0%} zeros")  # fmt: skip

    # --- normalise, choose highly variable genes, PCA ---
    scaled, X = normalise(counts)  # (C, G)
    hvg = torch.argsort(hvg_scores(scaled), descending=True)[: params["n_hvg"]]  # (H,) gene indices
    hit = torch.isin(hvg, markers.flatten()).float().mean().item()
    print(f"highly variable genes: {len(hvg)} kept, {hit:.0%} of them are true marker genes")
    n_pcs = params["n_pcs"]
    y, _, ratio = pca(X[:, hvg], n_pcs)  # (C, P), explained-variance ratio of all components
    print("explained variance PC1…PC6: " + "  ".join(f"{r:.1%}" for r in ratio[:6].tolist()))
    # Cross-check with the randomised SVD used for big data (a few extra components help it converge).
    _, S_low, _ = torch.pca_lowrank(X[:, hvg], q=n_pcs + 10, niter=6)
    S_full = torch.linalg.svdvals(X[:, hvg] - X[:, hvg].mean(0))
    gap = ((S_low[:n_pcs] - S_full[:n_pcs]).abs().max() / S_full[0]).item()
    print(f"torch.pca_lowrank: top {n_pcs} singular values match the full SVD to {gap:.1e} (relative)")

    # --- k-means in the space of the first n_pcs components ---
    km = solve.kmeans(y, params["k"], N_INIT, 4 if fast else params["kmeans_iters"], SEED)
    best_inertia = km["inertia"][:, km["best"]]
    ari = [adjusted_rand_index(lab, labels) for lab in km["labels"]]
    finals = ", ".join(f"{v:.0f}" for v in km["inertia"][-1])
    print(f"k-means (k = {params['k']}, {N_INIT} restarts at once): inertia {best_inertia[0]:.0f} → "
          f"{best_inertia[-1]:.0f}; final inertias of the restarts {finals}")  # fmt: skip
    print(f"adjusted Rand index vs true types: {ari[0]:.3f} after the first assignment → {ari[-1]:.3f}")

    # --- a 2D picture: PC1–PC2, or a t-SNE map ---
    if params["embedding"] == "tsne":
        n_steps = 100 if fast else 300
        emb = solve.tsne(y, 30.0, n_steps, SEED)
        print(f"t-SNE: {n_steps} Adam steps on the PC scores (perplexity 30)")
    else:
        emb = y[:, :2]
    print(f"total time {time.perf_counter() - t0:.1f} s")

    return plot.animate(
        y.cpu(), emb.cpu(), labels.cpu(), km, ari, ratio.cpu(), n_pcs, params["embedding"],
        out_dir / "single_cell_pca.gif",
    )  # fmt: skip


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/bioinformatics/single_cell_pca/equation.py
r"""Single-cell analysis → tensor translation.

Counts K_{cg}: cell c, gene g. The standard first steps of every single-cell RNA-seq analysis:

    n_c      = Σ_g K_{cg}                                   library size (g summed)
    X_{cg}   = log(1 + 10⁴ K_{cg} / n_c)                     depth normalisation + log
    z_g      = standardised log(Var_c X̃_{cg} / Mean_c X̃_{cg}) highly-variable-gene score (within bins of mean)
    Z_{cg}   = X_{cg} − ⟨X_{cg}⟩_c = U_{cp} S_p V_{gp}       centre, then SVD
    y_{cp}   = Z_{cg} V_{gp}                                 PC scores (g summed); explained variance S_p² / Σ S²

k-means in PC space, for R restarts r at once (p summed in the distances, c summed in the means):

    d_{rck}  = y_{cp}y_{cp} − 2 y_{cp}μ_{rkp} + μ_{rkp}μ_{rkp}
    A_{rck}  = 1 if k = argmin_k d_{rck}
    μ_{rkp}  = A_{rck} y_{cp} / A_{rck}
"""

import torch


def library_size(counts: torch.Tensor) -> torch.Tensor:
    """n_c = Σ_g K_{cg}.   (C, G) → (C,);  g summed."""
    return torch.einsum("cg->c", counts)


def normalise(counts: torch.Tensor, scale: float = 1e4) -> tuple[torch.Tensor, torch.Tensor]:
    """Depth-normalised counts X̃ = scale · K_{cg} / n_c (counts per 10⁴, the CPM idea) and X = log(1 + X̃)."""
    scaled = scale * counts / library_size(counts)[:, None]
    return scaled, torch.log1p(scaled)


def hvg_scores(scaled: torch.Tensor, n_bins: int = 20) -> torch.Tensor:
    """Highly-variable-gene score z_g (G,): the dispersion D_g = Var/Mean of the normalised counts, standardised
    among genes of similar mean expression.

    Noise alone makes D_g grow with expression (negative-binomial variance m + φm²), so genes are put into
    n_bins bins of log mean, and z_g = (log D_g − bin mean) / bin std.  The bin averages are one-hot sums:
    B_{gb} = 1 if gene g is in bin b,  ⟨log D⟩_b = B_{gb} log D_g / B_{gb}  (g summed).
    """
    mean = scaled.mean(0).clamp_min(1e-8)
    log_d = torch.log(scaled.var(0) / mean + 1e-12)  # (G,)
    log_m = torch.log(mean)
    edges = torch.quantile(log_m, torch.linspace(0, 1, n_bins + 1, device=scaled.device)[1:-1])
    bin_of = torch.bucketize(log_m, edges)  # (G,) bin index
    B = torch.nn.functional.one_hot(bin_of, n_bins).float()  # (G, n_bins)
    count = torch.einsum("gb->b", B)
    bin_mean = torch.einsum("gb,g->b", B, log_d) / count
    bin_std = (torch.einsum("gb,g->b", B, log_d**2) / count - bin_mean**2).clamp_min(1e-12).sqrt()
    return (log_d - bin_mean[bin_of]) / bin_std[bin_of]


def pca(X: torch.Tensor, n_pcs: int):
    """Centre the cells × genes matrix and take its SVD Z = U S Vᵀ.

    Returns the scores y_{cp} = Z_{cg} V_{gp} (C, P), the loadings V (G, P) and the explained-variance ratio
    S_p² / Σ_q S_q² of every component (sorted from largest to smallest, sums to 1 over all components).
    """
    Z = X - X.mean(0)  # (C, G)
    U, S, Vh = torch.linalg.svd(Z, full_matrices=False)
    V = Vh[:n_pcs].T  # (G, P)
    scores = torch.einsum("cg,gp->cp", Z, V)  # = U[:, :P] * S[:P]
    return scores, V, S**2 / (S**2).sum()


def sq_distances(y: torch.Tensor, mu: torch.Tensor) -> torch.Tensor:
    """d_{rck} = ‖y_c − μ_rk‖².   y: (C, P), mu: (R, K, P) → (R, C, K);  p summed."""
    yy = torch.einsum("cp,cp->c", y, y)
    ym = torch.einsum("cp,rkp->rck", y, mu)
    mm = torch.einsum("rkp,rkp->rk", mu, mu)
    return (yy[None, :, None] - 2 * ym + mm[:, None, :]).clamp_min(0.0)


def assign(d: torch.Tensor) -> torch.Tensor:
    """One-hot assignment A_{rck} to the nearest centroid.   (R, C, K) → (R, C, K)."""
    return torch.nn.functional.one_hot(d.argmin(-1), d.shape[-1]).float()


def update_centroids(y: torch.Tensor, A: torch.Tensor, mu_old: torch.Tensor) -> torch.Tensor:
    """μ_{rkp} = Σ_c A_{rck} y_{cp} / Σ_c A_{rck};  an empty cluster keeps its old centre.   → (R, K, P);  c summed."""
    total = torch.einsum("rck,cp->rkp", A, y)
    count = torch.einsum("rck->rk", A)[..., None]
    return torch.where(count > 0, total / count.clamp_min(1.0), mu_old)


def inertia(d: torch.Tensor, A: torch.Tensor) -> torch.Tensor:
    """Within-cluster sum of squares Σ_{c,k} A_{rck} d_{rck} for each restart.   → (R,)."""
    return torch.einsum("rck,rck->r", A, d)


def adjusted_rand_index(a: torch.Tensor, b: torch.Tensor) -> float:
    """ARI of two labelings (C,) via the contingency table n_{ij} = Σ_c [a_c = i][b_c = j]  (c summed).

    ARI = (Σ C(n_ij,2) − E) / (½(Σ C(a_i,2) + Σ C(b_j,2)) − E),  E = Σ C(a_i,2) Σ C(b_j,2) / C(n,2):
    1 for identical partitions (up to renaming), about 0 for random ones.
    """
    A = torch.nn.functional.one_hot(a).double()
    B = torch.nn.functional.one_hot(b).double()
    n = torch.einsum("ci,cj->ij", A, B)

    def pairs(x: torch.Tensor) -> torch.Tensor:
        return (x * (x - 1) / 2).sum()

    index, sum_a, sum_b = pairs(n), pairs(n.sum(1)), pairs(n.sum(0))
    expected = sum_a * sum_b / pairs(torch.tensor(float(len(a)), dtype=torch.float64))
    return ((index - expected) / (0.5 * (sum_a + sum_b) - expected)).item()


def tsne_affinities(y: torch.Tensor, perplexity: float, n_bisect: int = 40) -> torch.Tensor:
    """t-SNE input similarities P_{ij}: a Gaussian around each cell with its width tuned to the perplexity.

    p_{j|i} ∝ exp(−β_i ‖y_i − y_j‖²), with β_i found by bisection (all cells at once) so that the entropy is
    log(perplexity); then P = (p_{j|i} + p_{i|j}) / 2C.
    """
    C = y.shape[0]
    d = torch.cdist(y, y) ** 2  # (C, C)
    d.fill_diagonal_(float("inf"))
    lo = torch.full((C, 1), 1e-6, dtype=y.dtype, device=y.device)
    hi = torch.full((C, 1), 1e3, dtype=y.dtype, device=y.device)
    target = torch.log(torch.tensor(perplexity, dtype=y.dtype))
    for _ in range(n_bisect):
        beta = (lo * hi).sqrt()  # bisection on a log scale
        p = torch.softmax(-beta * d, dim=1)
        entropy = -(p * torch.log(p.clamp_min(1e-12))).sum(1, keepdim=True)
        too_wide = entropy > target  # too many neighbours → sharpen (larger β)
        lo = torch.where(too_wide, beta, lo)
        hi = torch.where(too_wide, hi, beta)
    p = torch.softmax(-(lo * hi).sqrt() * d, dim=1)
    return (p + p.T) / (2 * C)


def tsne_loss(Y: torch.Tensor, P: torch.Tensor, exaggeration: float = 1.0) -> torch.Tensor:
    """KL(P ‖ Q) up to a constant, −Σ_ij P_ij log Q_ij, with Student-t similarities in the 2D map:

    q_ij = w_ij / Σ_{k≠l} w_kl,   w_ij = 1 / (1 + d_ij),   d_ij = Y_ia Y_ia + Y_ja Y_ja − 2 Y_ia Y_ja  (a summed).
    Since Σ P = 1:  −Σ P log Q = Σ P_ij log(1 + d_ij) + log Σ w.   Early on P is multiplied by an
    "exaggeration" factor, which pulls clusters together before they settle.
    """
    sq = torch.einsum("ia,ia->i", Y, Y)
    d = (sq[:, None] + sq[None, :] - 2 * torch.einsum("ia,ja->ij", Y, Y)).clamp_min(0.0)  # (C, C)
    w_sum = (1.0 / (1.0 + d)).sum() - len(Y)  # leave out the diagonal (d_ii = 0, w_ii = 1)
    return exaggeration * ((P * torch.log1p(d)).sum() + torch.log(w_sum))


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/bioinformatics/single_cell_pca/tensors.py
"""Build the concrete tensors: a synthetic single-cell RNA-seq experiment with known cell types."""

import torch


def type_profiles(n_genes: int, n_types: int, n_markers: int, fold_change: float, seed: int, device: torch.device):
    """Mean expression μ_{kg} of gene g in cell type k, shape (K, G), and the marker genes of each type (K, M).

    Every gene has a baseline drawn from a log-normal (a few genes are very highly expressed, most are low);
    each type then switches on its own n_markers marker genes by a factor fold_change.
    """
    gen = torch.Generator().manual_seed(seed)
    base = torch.exp(torch.randn(n_genes, generator=gen) * 1.2 - 1.0)  # (G,) mean counts per cell at depth 1
    markers = torch.randperm(n_genes, generator=gen)[: n_types * n_markers].reshape(n_types, n_markers)  # disjoint sets
    mu = base.repeat(n_types, 1)  # (K, G)
    for k in range(n_types):
        mu[k, markers[k]] *= fold_change
    return mu.to(device), markers.to(device)


def sample_counts(n_cells: int, mu: torch.Tensor, dispersion: float, depth_spread: float, seed: int):
    """Draw a count matrix K_{cg}, shape (C, G), plus the true type of each cell (C,) and its size factor s_c (C,).

    Negative-binomial counts as a Gamma–Poisson mixture:  λ_{cg} ~ Gamma(shape 1/φ, mean s_c μ_{k(c) g}),
    K_{cg} ~ Poisson(λ_{cg}).  The variance is  m + φ m²  (overdispersed, as in real data).
    s_c ~ log-normal: cells are sequenced to different depths (library-size variation).
    """
    K, G = mu.shape
    dev = mu.device
    gen = torch.Generator().manual_seed(seed)
    labels = torch.randint(K, (n_cells,), generator=gen)
    s = torch.exp(depth_spread * torch.randn(n_cells, generator=gen))  # (C,) size factors, median 1
    mean = s[:, None] * mu.cpu()[labels]  # (C, G)  m_{cg} = s_c μ_{k(c) g}
    with torch.random.fork_rng(devices=[]):  # the Gamma sampler has no generator argument: seed a private stream
        torch.manual_seed(seed + 1)
        shape = torch.full_like(mean, 1.0 / dispersion)
        lam = torch.distributions.Gamma(shape, shape / mean).sample()  # mean = shape / rate = m_cg
    counts = torch.poisson(lam, generator=gen)
    return counts.to(dev), labels.to(dev), s.to(dev)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/bioinformatics/single_cell_pca/solve.py
"""k-means with several restarts at once (batched tensor ops), and a small t-SNE map trained with autograd."""

import torch

from problems._common.progress import steps

from .equation import assign, inertia, sq_distances, tsne_affinities, tsne_loss, update_centroids


def kmeans(y: torch.Tensor, k: int, n_init: int, n_iter: int, seed: int) -> dict:
    """Lloyd's algorithm for R = n_init random starts in parallel; keep the start with the lowest final inertia.

    Each start picks k random cells as its first centres. Every iteration: assign each cell to its nearest centre,
    then move each centre to the mean of its cells. Neither step can increase the inertia.
    Returns the best run's centres (T+1, K, P) and labels (T+1, C) per iteration, and all inertias (T+1, R).
    """
    gen = torch.Generator().manual_seed(seed)
    C = y.shape[0]
    starts = torch.stack([torch.randperm(C, generator=gen)[:k] for _ in range(n_init)]).to(y.device)  # (R, K)
    mu = y[starts]  # (R, K, P)
    centres, labels, inertias = [], [], []
    for it in steps(n_iter + 1, "k-means"):
        d = sq_distances(y, mu)  # (R, C, K)
        A = assign(d)  # (R, C, K)
        centres.append(mu)
        labels.append(d.argmin(-1))
        inertias.append(inertia(d, A))
        if it < n_iter:
            mu = update_centroids(y, A, mu)
    inertias = torch.stack(inertias)  # (T+1, R)
    best = int(inertias[-1].argmin())
    return {
        "centres": torch.stack(centres)[:, best],  # (T+1, K, P)
        "labels": torch.stack(labels)[:, best],  # (T+1, C)
        "inertia": inertias,  # (T+1, R)
        "best": best,
    }


def tsne(y: torch.Tensor, perplexity: float, n_iter: int, seed: int) -> torch.Tensor:
    """2D t-SNE map of the PC scores y (C, P), optimised with Adam; starts from PC1–PC2 (scaled small)."""
    P = tsne_affinities(y, perplexity)
    Y0 = y[:, :2] / y[:, :2].std() * 1e-2
    gen = torch.Generator().manual_seed(seed)
    Y = (Y0 + 1e-4 * torch.randn(Y0.shape, generator=gen).to(y.device)).requires_grad_(True)
    opt = torch.optim.Adam([Y], lr=1.0)
    for it in steps(n_iter, "t-SNE"):
        loss = tsne_loss(Y, P, exaggeration=12.0 if it < n_iter // 4 else 1.0)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return Y.detach()


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/bioinformatics/single_cell_pca/plot.py
"""Animate k-means in PC space next to the true cell types, the inertia and the explained variance."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif

BLUE, GREY = "#2563eb", "#9ca3af"


def match_colours(final: np.ndarray, truth: np.ndarray, k: int) -> np.ndarray:
    """Colour index for each cluster: the true type most of its cells belong to (so colours line up)."""
    return np.array([np.bincount(truth[final == j]).argmax() if (final == j).any() else j for j in range(k)])


def animate(
    y: torch.Tensor,
    emb: torch.Tensor,
    labels: torch.Tensor,
    km: dict,
    ari: list[float],
    ratio: torch.Tensor,
    n_pcs: int,
    embedding: str,
    path: Path,
) -> Path:
    pc = y[:, :2].numpy()
    e = emb.numpy()
    truth = labels.numpy()
    centres = km["centres"][:, :, :2].cpu().numpy()  # (T+1, K, 2)
    assign = km["labels"].cpu().numpy()  # (T+1, C)
    inertia = km["inertia"].cpu().numpy()  # (T+1, R)
    T1, k = centres.shape[:2]
    cmap = np.array(ACCENT)
    colour_of = cmap[match_colours(assign[-1], truth, k) % len(cmap)]  # (K,) colour per cluster

    fig = plt.figure(figsize=(9.6, 4.0))
    grid = fig.add_gridspec(2, 3, width_ratios=[1.3, 1.1, 1.0])
    ax = fig.add_subplot(grid[:, 0])
    ax_t = fig.add_subplot(grid[:, 1])
    ax_i = fig.add_subplot(grid[0, 2])
    ax_v = fig.add_subplot(grid[1, 2])

    # --- k-means in PC1–PC2 ---
    dots = ax.scatter(pc[:, 0], pc[:, 1], s=6, c=colour_of[assign[0]], lw=0, alpha=0.8)
    stars = ax.scatter(centres[0, :, 0], centres[0, :, 1], s=160, marker="X", c=colour_of, edgecolors="#111827", lw=1.2)
    trails = [ax.plot([], [], color="#111827", lw=1, alpha=0.6)[0] for _ in range(k)]
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title(f"k-means on {n_pcs} PCs (shown: PC1–PC2)", fontsize=10)
    counter = ax.text(0.02, 0.97, "", transform=ax.transAxes, fontsize=9, va="top")

    # --- the true cell types ---
    for t in np.unique(truth):
        sel = truth == t
        ax_t.scatter(e[sel, 0], e[sel, 1], s=6, color=cmap[t % len(cmap)], lw=0, alpha=0.8, label=f"type {t + 1}")
    ax_t.legend(loc="best", fontsize=7, markerscale=2.5, framealpha=0.9)
    name = "t-SNE" if embedding == "tsne" else "PC1–PC2"
    ax_t.set_title(f"true cell types ({name})", fontsize=10)
    ax_t.set_xticks([])
    ax_t.set_yticks([])

    # --- inertia per iteration: all restarts (grey), the kept one (blue) ---
    best = km["best"]
    its = np.arange(T1)
    for r in range(inertia.shape[1]):
        if r != best:
            ax_i.plot(its, inertia[:, r], color=GREY, lw=0.8, alpha=0.7)
    (line_i,) = ax_i.plot([], [], "o-", color=BLUE, ms=3, lw=1.6)
    ax_i.set_xlim(0, T1 - 1)
    ax_i.set_title("inertia (within-cluster SS)", fontsize=9)
    ax_i.tick_params(labelsize=7)

    # --- scree plot ---
    n_bar = min(10, len(ratio))
    ax_v.bar(np.arange(1, n_bar + 1), 100 * ratio[:n_bar].numpy(), color=[BLUE if p < n_pcs else GREY
             for p in range(n_bar)])  # fmt: skip
    ax_v.set_title("explained variance per PC (%)", fontsize=9)
    ax_v.set_xticks(np.arange(1, n_bar + 1))
    ax_v.tick_params(labelsize=7)
    ax_v.set_xlabel("component", fontsize=8)

    def update(t: int):
        dots.set_color(colour_of[assign[t]])
        stars.set_offsets(centres[t])
        for j, trail in enumerate(trails):
            trail.set_data(centres[: t + 1, j, 0], centres[: t + 1, j, 1])
        line_i.set_data(its[: t + 1], inertia[: t + 1, best])
        counter.set_text(f"iteration {t}   ARI = {ari[t]:.3f}")

    fig.tight_layout()
    steps_shown = frame_indices(T1, 40)
    frames = [t for t in steps_shown for _ in range(4)]  # hold each iteration for a few frames
    return save_gif(fig, update, frames, path, fps=8)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_cells': 1500,
    'n_genes': 2000,
    'n_types': 4,
    'n_markers': 40,
    'fold_change': 3.0,
    'dispersion': 0.3,
    'depth_spread': 0.5,
    'n_hvg': 200,
    'n_pcs': 10,
    'k': 4,
    'kmeans_iters': 12,
    'embedding': 'pca'
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.bioinformatics.single_cell_pca")]:
    del sys.modules[name]
main = importlib.import_module("problems.bioinformatics.single_cell_pca.main")
problem_dir = Path("problems/bioinformatics/single_cell_pca")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
